# 10 — Validating honestly: cross-validation

*Module 00 · Getting Started — notebook 10 of 11*

Notebook 09 left one honest loose end: we chose the polynomial degree by reading the **test** error. This notebook gives you a way to choose such a setting **without ever touching the test set**, so the final number you report stays trustworthy.

**Prerequisites:** 04 (the train/test split), 09 (we picked the degree on the test curve — the loose end we close here).

**What you'll be able to do**
- Tell a **parameter** (learned by `fit`) from a **hyperparameter** (set by you, like the degree).
- Explain why choosing a setting on the test set makes your reported score too optimistic.
- Build **stratified k-fold cross-validation** by hand and use it to choose the degree.
- Refit and report a single, honest estimate on the sealed test set.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.patches import Patch
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from ml_course import colors, viz

np.random.seed(0)
viz.use_course_style()


def flexible_boundary(degree):
    '''A classifier whose complexity is set by the polynomial degree (as in notebook 09).

    Plumbing only: expand the two features to polynomials of the given degree, put them on a common
    scale, and fit a plain linear classifier (logistic regression — chapter 03; scaling — notebook
    11). The one knob we turn is the degree.
    '''
    return make_pipeline(
        PolynomialFeatures(degree),
        StandardScaler(),
        LogisticRegression(C=1e6, max_iter=5000),
    )


# The same harder problem as notebook 09: two interleaving moons with noise.
X, y = make_moons(n_samples=300, noise=0.30, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=0, stratify=y
)
print(f'training examples: {len(X_train)}    test examples (sealed): {len(X_test)}')

## Where we are, and the loose end

From notebook 04: we split the data once into a **training** set (to fit on) and a **test** set (a sealed sample of unseen data, used to estimate how well the model generalizes). The rule was strict — the test set is looked at once, at the very end.

In notebook 09 we bent that rule. To find the best polynomial degree we tried every degree, read the **test** error for each, and kept the degree with the lowest test error. It worked, but it spent the test set: once we pick the option that looks best on those particular points, the test score for that option is no longer an honest estimate of performance on genuinely new data. We adapted a choice to the test set, so the test set can no longer judge it impartially.

We need a way to choose the degree using only the training data. That is what this notebook builds.

## Parameters vs hyperparameters

It helps to separate two kinds of numbers in a model.

- **Parameters** are learned from the data by `fit` — here, the weights of the linear classifier inside `flexible_boundary`. You never set these by hand; fitting does.
- **Hyperparameters** are set by *you*, before fitting, and they shape what fitting can do — here, the **polynomial degree**. Fitting will happily learn weights for whatever degree you hand it, but it will not tell you which degree to use.

Choosing the degree is therefore our job, not `fit`'s. The question is how to do it honestly.

## A third slice: the validation set

The fix is to keep the test set sealed and carve a *new* slice out of the training data to make the choice on. Now there are three slices, each with one job:

- **training** — fit the parameters (the weights), for each candidate degree;
- **validation** — score those fitted models and pick the degree;
- **test** — sealed until the very end, for one final honest estimate.

The degree is chosen by looking only at validation, never at test. Let us try it.

In [ ]:
# Carve a validation slice out of the TRAINING data; the test set stays sealed.
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=0, stratify=y_train
)
print(f'fit on {len(X_fit)} points, validate on {len(X_val)}, test sealed: {len(X_test)}')

degrees = list(range(1, 10))
train_err, val_err = [], []
for degree in degrees:
    model = flexible_boundary(degree).fit(X_fit, y_fit)
    train_err.append(1 - model.score(X_fit, y_fit))
    val_err.append(1 - model.score(X_val, y_val))

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(degrees, train_err, color=colors.COLORS['train'], marker='o', linewidth=2, label='training')
ax.plot(degrees, val_err, color=colors.COLORS['test'], marker='s', linewidth=2, label='validation')
ax.set_xlabel('polynomial degree')
ax.set_ylabel('error rate')
ax.legend()
plt.show()

val_pick = degrees[int(np.argmin(val_err))]
print(f'degree picked on the validation set: {val_pick}')

### Read the figure

For each degree we fit on the training slice and measured the error on the **validation** slice. The training error drifts down as the degree rises (a more flexible model fits its own data more closely), while the validation error traces the familiar **U** — too rigid on the left, too wiggly on the right, lowest in between. We read the degree at the bottom of the validation U and pick it, and we did this **without touching the test set**. The validation slice played the role notebook 09 wrongly handed to the test set.

## The catch with a single split

Two things should make us uneasy about that one validation split.

- We **spent data**: the points sitting in the validation slice were taken away from training, so every candidate model was fitted on less data than we actually have.
- The choice is **unstable**: which points happened to land in the validation slice was the luck of one random split. A different split could hand us a different best degree.

The second point is worth seeing for ourselves.

In [ ]:
print('Degree chosen from a single validation split, across several random splits:')
for seed in range(6):
    Xa, Xb, ya, yb = train_test_split(
        X_train, y_train, test_size=0.25, random_state=seed, stratify=y_train
    )
    seed_val_err = [1 - flexible_boundary(d).fit(Xa, ya).score(Xb, yb) for d in degrees]
    print(f'  seed {seed}:  degree {degrees[int(np.argmin(seed_val_err))]}')

### What that tells us

The chosen degree jumps around as the split changes — the single validation estimate is noisy, built from one small slice. We would like to use **all** of the training data for validating, and to average over several splits so the choice stops depending on one unlucky cut. That is exactly what k-fold cross-validation does.

## k-fold cross-validation

Here is the idea. Split the training data into **k** equal parts, called **folds**. Then go around k times: each time, one fold is held out as the validation slice while the other k-1 folds are trained on. Every fold serves as validation exactly once, and every point gets validated — on a model that never saw it. Averaging the k validation scores gives a steadier estimate than any single split.

One detail matters for classification: we want each fold to hold the **same class proportions** as the whole set, so no fold is accidentally short on one class. Keeping the proportions per fold is called **stratification** — the same idea as the stratified train/test split from notebook 04.

In [ ]:
k = 5
fig, ax = plt.subplots(figsize=(8, 3.8))
for round_idx in range(k):
    for block in range(k):
        is_val = block == round_idx
        ax.barh(
            round_idx, 1, left=block, height=0.8,
            color=colors.COLORS['test'] if is_val else colors.COLORS['train'],
            edgecolor=colors.COLORS['background'], linewidth=1.5,
        )
    ax.text(-0.15, round_idx, f'round {round_idx + 1}', ha='right', va='center',
            color=colors.COLORS['text'])
ax.set_xlim(-1.3, k)
ax.invert_yaxis()
ax.set_xticks([b + 0.5 for b in range(k)], [f'fold {b + 1}' for b in range(k)])
ax.set_yticks([])
ax.set_title('5-fold cross-validation: each fold is the validation set once')
ax.legend(
    handles=[Patch(color=colors.COLORS['train'], label='train'),
             Patch(color=colors.COLORS['test'], label='validation')],
    loc='upper center', bbox_to_anchor=(0.5, 1.28), ncol=2,
)
ax.grid(False)
plt.show()

### Read the figure

Each row is one round of cross-validation. The highlighted block is the fold held out for validation that round; the rest are trained on. Reading down the rows, the validation block marches across all five folds — so across the five rounds, every point is validated exactly once and trained on four times. Because we will build the folds with stratification, each validation block carries the same class balance as the full set.

In [ ]:
def stratified_kfold(labels, n_splits=5, seed=0):
    '''Return a list of (train_idx, val_idx) pairs that preserve each class's proportion.

    For every class we shuffle that class's positions and cut them into n_splits nearly equal
    chunks; fold f takes chunk f from every class as its validation indices. Because each class is
    chunked separately, every fold matches the class proportions of the whole set as closely as
    whole-number fold sizes allow (stratification) — exactly when each class divides evenly by
    n_splits, approximately otherwise. We use np.array_split (not np.split) so a class whose size is
    not a multiple of n_splits, or even smaller than n_splits, still splits without raising.
    '''
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)
    per_class_chunks = {}
    for cls in np.unique(labels):
        idx = np.where(labels == cls)[0]
        rng.shuffle(idx)
        per_class_chunks[cls] = np.array_split(idx, n_splits)
    folds = []
    for f in range(n_splits):
        val_idx = np.concatenate([per_class_chunks[cls][f] for cls in per_class_chunks])
        train_idx = np.setdiff1d(np.arange(len(labels)), val_idx)
        folds.append((np.sort(train_idx), np.sort(val_idx)))
    return folds


folds = stratified_kfold(y_train, n_splits=5, seed=0)
for f, (train_idx, val_idx) in enumerate(folds):
    val_class1_share = y_train[val_idx].mean()
    print(f'fold {f + 1}:  train {len(train_idx):3d}   val {len(val_idx):3d}   '
          f'class-1 share in val {val_class1_share:.2f}')
print(f'class-1 share over the whole training set: {y_train.mean():.2f}')

### Read the output

The five folds are the same size, and each validation block carries the same class-1 share as the full training set. Two honest notes. First, the shares come out *identical* here because the two classes are equal in size and divide evenly into five; with uneven classes, stratification matches the overall balance as closely as whole-number fold sizes allow, not exactly. Second, on this 50/50 set a plain random split would also land near 0.50 — so what we see is stratification *guaranteeing* the balance, not visibly rescuing it; notebook 04 showed it biting on the less balanced penguins. These `(train, validation)` index pairs are all the machinery k-fold needs; everything below loops over them.

In [ ]:
records = []
cv_train_err, cv_val_err = [], []
for degree in degrees:
    train_scores, val_scores = [], []
    for train_idx, val_idx in folds:
        model = flexible_boundary(degree).fit(X_train[train_idx], y_train[train_idx])
        train_scores.append(model.score(X_train[train_idx], y_train[train_idx]))
        val_scores.append(model.score(X_train[val_idx], y_train[val_idx]))
    cv_train_err.append(1 - np.mean(train_scores))
    cv_val_err.append(1 - np.mean(val_scores))
    row = {'degree': degree}
    for i, s in enumerate(val_scores):
        row[f'fold{i + 1}'] = round(s, 3)
    row['mean_val_acc'] = round(float(np.mean(val_scores)), 4)
    records.append(row)

cv_table = pd.DataFrame(records).set_index('degree')
cv_pick = degrees[int(np.argmin(cv_val_err))]
display(cv_table)
print(f'degree chosen by 5-fold cross-validation: {cv_pick}')

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(degrees, cv_train_err, color=colors.COLORS['train'], marker='o', linewidth=2,
        label='training (CV)')
ax.plot(degrees, cv_val_err, color=colors.COLORS['test'], marker='s', linewidth=2,
        label='validation (CV)')
ax.axvline(cv_pick, color=colors.COLORS['highlight'], linestyle='--', linewidth=1.2,
           label=f'chosen degree = {cv_pick}')
ax.set_xlabel('polynomial degree')
ax.set_ylabel('error rate')
ax.legend()
plt.show()

### Read the figure

This is the same kind of curve as before, but now each point is an **average over five folds**, so it wobbles far less than a single split did. The cross-validation error dips to its minimum at **degree 3** and rises on either side; that is the degree we choose — using the training data alone. Reassuringly, it agrees with the sweet spot notebook 09 spotted by eye on the test curve, except this time the test set never entered the decision.

In [ ]:
# cross_val_score is the library version of the by-hand loop above. Fed the SAME folds, it must
# return the same numbers.
by_hand_mean = np.mean([
    flexible_boundary(cv_pick).fit(X_train[tr], y_train[tr]).score(X_train[va], y_train[va])
    for tr, va in folds
])
sklearn_mean = cross_val_score(flexible_boundary(cv_pick), X_train, y_train, cv=folds).mean()
print(f'by-hand CV mean (degree {cv_pick}):      {by_hand_mean:.6f}')
print(f'cross_val_score on the same folds:    {sklearn_mean:.6f}')
print(f'identical: {np.isclose(by_hand_mean, sklearn_mean)}')

### Read the output

The two numbers match to the digit — and they match *because* we handed `cross_val_score` the very folds we built (through `cv=folds`), so it walks the identical index pairs and averages identically. It is not doing anything mysterious; it runs exactly the loop we wrote by hand. (Had we passed `cv=5` instead, scikit-learn would build its own reshuffled folds and land slightly differently — not wrong, only a different partition.) scikit-learn's `StratifiedKFold` is the ready-made version of our `stratified_kfold`, building the same kind of class-balanced folds for you. Now that we trust it, we lean on the library for the final experiment.

## The one honest estimate

Cross-validation chose the degree using only the training data, and the test set is still sealed. Now — and only now — we do the thing we have been saving it for: refit the chosen model on **all** of the training data and score it **once** on the test set. That single number is our honest estimate of how this model behaves on new data.

In [ ]:
final_model = flexible_boundary(cv_pick).fit(X_train, y_train)
test_accuracy = final_model.score(X_test, y_test)
print(f'chosen degree (by cross-validation): {cv_pick}')
print(f'final test accuracy (scored once):   {test_accuracy:.4f}')

## Why not pick the degree on the test set?

We keep insisting on sealing the test set. Here is the cost of not doing so. Suppose we skip cross-validation and instead try all nine degrees directly on the test set, then report the best test score we saw. By construction that best-of-nine number flatters us: with nine tries on the same fixed sample, one of them lands well partly by luck, and we reported precisely that one. The test set stopped being a fair, untouched sample the moment we used it to choose.

Let us measure how big that optimistic bias is, by averaging over many random splits.

In [ ]:
# Two protocols, compared across 100 random train/test splits:
#   honest  -> choose the degree by cross-validation on the training set, then score test once
#   cheat   -> choose the degree by its test score, then report that score
# This refits many models, so it takes a few seconds.
honest, cheat = [], []
for seed in range(100):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.30, random_state=seed, stratify=y)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    cv_means = [cross_val_score(flexible_boundary(d), Xa, ya, cv=cv).mean() for d in degrees]
    degree_by_cv = degrees[int(np.argmax(cv_means))]
    honest.append(flexible_boundary(degree_by_cv).fit(Xa, ya).score(Xb, yb))
    cheat.append(max(flexible_boundary(d).fit(Xa, ya).score(Xb, yb) for d in degrees))

print(f'mean test accuracy, honest (CV chooses):     {np.mean(honest):.4f}')
print(f'mean test accuracy, tuning on the test set:  {np.mean(cheat):.4f}')
print(f'optimistic inflation: {np.mean(cheat) - np.mean(honest):+.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
labels = ['honest\n(CV chooses the degree)', 'tuning on test\n(keep the best test score)']
means = [float(np.mean(honest)), float(np.mean(cheat))]
bars = ax.bar(labels, means, color=[colors.COLORS['model'], colors.COLORS['highlight']],
              edgecolor=colors.COLORS['text'], linewidth=0.6)
for bar, m in zip(bars, means, strict=True):
    ax.text(bar.get_x() + bar.get_width() / 2, m, f'{m:.3f}', ha='center', va='bottom',
            color=colors.COLORS['text'])
ax.set_ylabel('mean test accuracy over 100 splits')
ax.set_ylim(0.85, 0.96)
ax.grid(False)
plt.show()

### Read the figure

Tuning on the test set reports a higher accuracy than the honest protocol — not because the model is better, but because we let the test set pick the winner and then quoted the winner's score. The gap here is small (about a point and a half), and it grows with the number of settings you try on the test set. On any single split the two protocols can land on the same degree and the gap vanishes; the bias only shows up as an average over many splits. That is why we touch the test set once, at the very end, after every choice has already been made.

## Your turn

1. You have 1000 training examples and run 5-fold cross-validation to evaluate one degree. How many models get fitted, and roughly how many examples does each one train on?
2. With very little data, people often raise k toward its extreme — k equal to the number of examples, so each fold validates on a single point (this is called *leave-one-out*). What does a larger k buy you, and what does it cost?
3. A colleague reports 99% accuracy on the test set after trying 50 different settings and keeping the best test score. What is wrong with that number, and what should they have done instead?

## What you built

You closed notebook 09's loose end and learned to choose a model setting honestly.

- You can separate a **parameter** (learned by `fit`) from a **hyperparameter** (chosen by you, like the degree).
- You built **stratified k-fold cross-validation** by hand, checked it against `cross_val_score`, and used it to choose the degree using the training data alone.
- You saw why a single validation split is noisy, and why averaging over folds steadies the choice.
- You reported one honest test-set estimate, and measured how much **tuning on the test set** inflates the number you would otherwise quote.

**Vocabulary added:** parameter vs hyperparameter, validation set, k-fold cross-validation, stratification, model selection, tuning-on-test inflation.

## Going further (optional)

- **Nested cross-validation.** We used CV to *choose* the degree and the test set to *estimate* performance. When you want to estimate the performance of the whole choose-then-fit procedure without a separate test set, you nest a second CV loop inside the first — more faithful, more expensive.
- **Leave-one-out (LOOCV).** k as large as the number of examples: each point is validated alone. Almost unbiased, but it refits the model n times and its estimate can be highly variable.
- **CV has variance too.** The folds overlap (they share training points), so the k fold-scores are correlated and their average is itself one noisy estimate — a reason to repeat CV with different seeds when a decision is close.

None of this is needed for the next notebook; it is here for the curious.

## References

- G. James, D. Witten, T. Hastie, R. Tibshirani, *An Introduction to Statistical Learning*, 2nd ed., Springer, 2021 — §5.1 (cross-validation: the validation-set approach, leave-one-out, and k-fold). DOI: 10.1007/978-1-0716-1418-1
- R. Kohavi (1995), *A study of cross-validation and bootstrap for accuracy estimation and model selection*, Proceedings of the 14th International Joint Conference on Artificial Intelligence (IJCAI), pp. 1137–1143 — the case for stratified k-fold in model selection.
- A. Géron, *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd ed., O'Reilly, 2022 — chapter 2 (cross-validation in practice).

---
Previous: **09 — Over-/under-fitting and the generalization gap** · Next: **11 — Preprocessing & leakage: scaling, encoding, Pipeline**